<a href="https://colab.research.google.com/github/dxda6216/ttron2excel/blob/main/ttron_data_file_to_excel_file_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io
import csv
import math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from scipy import signal
from scipy.optimize import curve_fit
from google.colab import files

#@title Converting Taylortron TRACES file (TRACES.nnn) to Excel file
#@markdown **This script works only with a specific format of data files (*TRACES.nnn* files) generated by the Taylortron.**

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def find_peaks_and_troughs(series, distance_in_hours=None, time_interval=None):
    """Detects peaks and troughs in a pandas Series."""
    if series.isnull().all():
        return [], []

    data_to_analyze = series.dropna().values
    if distance_in_hours and time_interval:
        distance_points = max(1, int(distance_in_hours / time_interval))
    else:
        distance_points = 5

    peaks, _ = signal.find_peaks(data_to_analyze, distance=distance_points)
    troughs, _ = signal.find_peaks(-data_to_analyze, distance=distance_points)

    original_indices = series.dropna().index
    return original_indices[peaks], original_indices[troughs]

def plot_channel_subplot(
    ax, x_scatter, y_scatter, scatter_label, scatter_color,
    x_line, y_line, line_label, line_color,
    x_lim_min, x_lim_max, y_lim_tuple,
    xticks_list, mit_on, mit_val, chlabel, current_plot_title=None
):
    """Plots a single channel subplot with raw/detrended data and trend lines."""
    ax.scatter(x_scatter, y_scatter, s=0.1, c=scatter_color, label=scatter_label)
    ax.plot(x_line, y_line, line_color, linewidth=0.5, label=line_label)
    ax.set_xlim(x_lim_min, x_lim_max)
    if y_lim_tuple:
        ax.set_ylim(y_lim_tuple[0], y_lim_tuple[1])
    ax.set_xticks(xticks_list)
    if mit_on:
        ax.xaxis.set_minor_locator(MultipleLocator(mit_val))
    ax.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
    ax.legend(loc='upper right', fontsize=4)
    if current_plot_title:
        ax.set_title(current_plot_title, fontsize=6)

def plot_individual_channel_summary(
    channelnumber, df, df_TL_MA, dtdf, dtdf_9PMA, Detrending_Method,
    Sinc_Filter_Cutoff_Period_Hours, filter_order,
    x_scale_min, x_scale_max, xtickslist, miton, mit,
    sub2label, sub3label, lod, last_time_point, time_interval, pp
):
    """Generates a 3-panel individual channel summary (Raw, Detrended, Actogram) and saves it to PDF."""
    fig = plt.figure(figsize=(8.5, 11))
    fig.suptitle(f"{Experiment_number}   Ch # {channelnumber}", fontsize=12)

    # --- Subplot 1: Raw Data with Trend Line ---
    ax1 = plt.subplot(3, 1, 1)
    x, y = df['Hours'], df[channelnumber]
    x_cma, y_cma = df_TL_MA['Hours'], df_TL_MA[channelnumber]

    ax1.scatter(x, y, s=3.0, c='violet', label='Bioluminescence')
    trend_lbl = f'Trend line (Sinc Filter C={Sinc_Filter_Cutoff_Period_Hours}h O={filter_order})' if Detrending_Method == 'Sinc Filter' else f'Trend line ({twss}-point MA, centered)'
    ax1.plot(x_cma, y_cma, '-r', linewidth=1.0, label=trend_lbl)
    ax1.set_xlim(x_scale_min, x_scale_max)
    ax1.set_ylim(0, int(max(y) * 1.1))
    ax1.set_xticks(xtickslist)
    if miton:
        ax1.xaxis.set_minor_locator(MultipleLocator(mit))
    ax1.set_xlabel('Hours', fontsize=10)
    ax1.set_ylabel('Bioluminescence', fontsize=10)
    ax1.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
    ax1.legend(loc='upper right', fontsize=5)

    # --- Subplot 2: Detrended Data with Smoothed Line and Peaks/Troughs ---
    ax2 = plt.subplot(3, 1, 2)
    x_dt, y_dt = dtdf['Hours'], dtdf[channelnumber]
    x_dt_cma, y_dt_cma = dtdf_9PMA['Hours'], dtdf_9PMA[channelnumber]

    ax2.scatter(x_dt, y_dt, s=3.0, c='violet', label='Detrended bioluminescence')
    ax2.plot(x_dt_cma, y_dt_cma, '-b', linewidth=1.0, label='Smoothed line (9PMA)')

    peaks_idx, troughs_idx = find_peaks_and_troughs(dtdf_9PMA[channelnumber], distance_in_hours=12, time_interval=time_interval)

    for idxs, marker, color, lbl, shift in [(peaks_idx, 'o', 'red', 'Peaks', 1), (troughs_idx, 'o', 'blue', 'Troughs', -1)]:
        if len(idxs) > 0:
            ax2.scatter(dtdf_9PMA.loc[idxs, 'Hours'], dtdf_9PMA.loc[idxs, channelnumber], marker=marker, s=30, color=color, label=lbl)
            if sub2label:
                for idx in idxs:
                    t_val = dtdf_9PMA.loc[idx, 'Hours']
                    v_val = dtdf_9PMA.loc[idx, channelnumber]
                    ax2.text(t_val, v_val + shift * (ax2.get_ylim()[1] - ax2.get_ylim()[0]) * 0.05, f'{t_val:.2f} h', fontsize=5, color=color, ha='center')

    ax2.set_xlim(x_scale_min, x_scale_max)
    ax2.set_xticks(xtickslist)
    if miton:
        ax2.xaxis.set_minor_locator(MultipleLocator(mit))
    ax2.set_xlabel('Hours', fontsize=10)
    ax2.set_ylabel('Detrended bioluminescence', fontsize=10)
    ax2.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
    ax2.legend(loc='upper right', fontsize=5)

    # --- Subplot 3: Actogram of Peaks and Troughs ---
    ax3 = plt.subplot(3, 1, 3)
    for idxs, color, lbl in [(peaks_idx, 'red', 'Peaks'), (troughs_idx, 'blue', 'Troughs')]:
        if len(idxs) > 0:
            hours = dtdf_9PMA.loc[idxs, 'Hours'].values
            act_data = []
            for h in hours:
                day_num = h // lod
                time_in_day = h - day_num * lod
                act_data.append((time_in_day, day_num, h))
                if day_num > 0:
                    act_data.append((time_in_day + lod, day_num - 1, h))
            x_act, y_act, orig_h = zip(*act_data)
            ax3.scatter(x_act, y_act, marker='o', s=20, color=color, label=lbl)
            if sub3label:
                for i in range(len(x_act)):
                    ax3.text(x_act[i] + 0.5, y_act[i], f'{orig_h[i]:.2f}', fontsize=5, color=color)

    ax3.set_xlim(-0.085 * lod if sub3label else 0, 2.085 * lod if sub3label else lod * 2)
    ax3.set_xticks([0, 0.25*lod, 0.5*lod, 0.75*lod, lod, 1.25*lod, 1.5*lod, 1.75*lod, 2*lod])
    ax3.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    ax3.set_title(f'Peaks and Troughs (T = {lod}h)' if lod != 24 else 'Peaks and Troughs', fontsize=10)
    ax3.set_ylabel('Days' if lod == 24 else f'Days (T{lod})', fontsize=10)

    ltpd = last_time_point // lod
    ax3.set_ylim(-ltpd*0.05, ltpd*1.05)
    ax3.yaxis.set_major_locator(MultipleLocator(1))
    ax3.set_xlabel('Time (Hours)', fontsize=10)
    ax3.grid(True, linestyle='--', alpha=0.7)
    ax3.legend(loc='upper right', fontsize=6)
    ax3.invert_yaxis()

    plt.tight_layout(rect=[0, 0.02, 1, 0.98])
    pp.savefig(fig)
    plt.show()  # Display the plot in Colab output before closing
    plt.close(fig)

def damped_sine_model(t, amplitude, period, phase, decay_rate, offset):
    return amplitude * np.exp(-decay_rate * t) * np.sin(2 * np.pi * t / period + phase) + offset

# ==============================================================================
# PARAMETER SELECTION & UI CONFIGURATION
# ==============================================================================
Experiment_number = 'CYxxx' #@param {type:"string"}
Experiment_title = '' #@param {type:"string"}
Date_experiment_started = '2026-01-01' #@param {type:"date"}
Detrending_Method = "Sinc Filter" #@param ["Sinc Filter", "Moving Average"]

Sinc_Filter_Cutoff_Period_Hours = 48 # @param {type:"slider", min:1, max:240, step:1}
Sinc_Filter_Order = 101 # @param {type:"slider", min:1, max:361, step:2}
Window_size_for_trend_line_moving_average = 24 # @param {type:"slider", min:1, max:120, step:1}

Data_Plotting = "Plotting the channel 00 data last" #@param ["Plotting the channel 00 data first", "Plotting the channel 00 data last"]
chlist = list(range(1, 30)) + [0] if Data_Plotting == "Plotting the channel 00 data last" else list(range(30))

Subplot_Matrix = "3-column by 10-row" #@param ["3-column by 10-row", "4-column by 8-row"]
subp_raw, subp_col = (10, 3) if Subplot_Matrix == "3-column by 10-row" else (8, 4)

Major_Ticks = "Every 24 hours" #@param ["Every 12 hours", "Every 24 hours", "Every 48 hours"]
mjt = {"Every 12 hours": 12, "Every 24 hours": 24, "Every 48 hours": 48}.get(Major_Ticks)

Minor_Ticks = "Every 12 hours" #@param ["No minor ticks", "Every 2 hours", "Every 4 hours", "Every 6 hours", "Every 12 hours", "Every 24 hours"]
mit = {"No minor ticks": 0, "Every 2 hours": 2, "Every 4 hours": 4, "Every 6 hours": 6, "Every 12 hours": 12, "Every 24 hours": 24}.get(Minor_Ticks)
miton = False if mit == 0 or mit >= mjt else True

Label_peaks_and_troughs_in_detrended_data_plot = "Yes" #@param ["Yes", "No"]
sub2label = (Label_peaks_and_troughs_in_detrended_data_plot == "Yes")

Actogram_X_axis_scale = 24 # @param {type:"slider", min:12, max:60, step:0.1}
lod = Actogram_X_axis_scale
Label_peaks_and_troughs_in_actogram = "Yes" #@param ["Yes", "No"]
sub3label = (Label_peaks_and_troughs_in_actogram == "Yes")

Start_Hour_for_Fitting = 24 #@param {type:"slider", min:0.0, max:360.0, step:1}
End_Hour_for_Fitting = 120 #@param {type:"slider", min:0.0, max:360.0, step:1}
if End_Hour_for_Fitting - Start_Hour_for_Fitting <= 23:
    Start_Hour_for_Fitting, End_Hour_for_Fitting = 24, 120

# ==============================================================================
# DATA LOADING & INITIAL CLEANUP
# ==============================================================================
!rm -rf *.xlsx *.pdf *.dat *.zip TRACES.* Traces.* traces.*
uploaded = files.upload()
ttronfilename = next(iter(uploaded))

sttime = datetime.now(timezone.utc)
print(f'\nStarted at {sttime.strftime("%Y-%m-%d %H:%M:%S")} (UTC)')

print('\nReading the data...')
colnames = ["Hours"] + [str(k).zfill(2) for k in range(30)]
df = pd.read_csv(ttronfilename, header=None, sep='\t', skiprows=3, skipfooter=1, index_col=False, names=colnames, engine='python')
df2 = df.iloc[:, 0:31]

display(df.head())
number_of_rows = len(df)
last_row_index = number_of_rows - 1
total_time = df.loc[last_row_index, 'Hours'] - df.loc[0, 'Hours']
time_interval = total_time / last_row_index
excel2_sheet_name = f'INTVL = {time_interval:.15f} h'
last_time_point = df.loc[last_row_index, 'Hours']

# ==============================================================================
# DETRENDING PROCESS
# ==============================================================================
print(f'\nCalculating trend line using {Detrending_Method}...')
df_5PMA = df.rolling(window=5, center=True, min_periods=1).mean()
df_9PMA = df.rolling(window=9, center=True, min_periods=1).mean()

if Detrending_Method == "Moving Average":
    tws = math.ceil(Window_size_for_trend_line_moving_average / time_interval)
    if tws % 2 == 0: tws += 1
    twss = int(tws)
    df_TL_MA = df.rolling(window=twss, center=True, min_periods=1).mean().fillna(method='bfill').fillna(method='ffill')
    trendline_sheet_name = f'Trend line ({twss}PMA)'
else:
    sampling_rate = 1 / time_interval
    norm_cutoff = (1 / Sinc_Filter_Cutoff_Period_Hours) / (0.5 * sampling_rate)
    filter_order = Sinc_Filter_Order
    max_order = int(number_of_rows / 3) - 1
    if filter_order >= max_order:
        filter_order = max(1, max_order - (1 if max_order % 2 == 0 else 0))
    if filter_order % 2 == 0: filter_order += 1

    sinc_filter_coeffs = signal.firwin(filter_order, norm_cutoff, pass_zero='lowpass')
    df_TL_MA = df.copy()
    for k in range(30):
        ch = str(k).zfill(2)
        if df[ch].isnull().all():
            df_TL_MA[ch] = np.nan
            continue
        interpolated = df[ch].interpolate(method='linear', limit_direction='both', axis=0)
        try:
            df_TL_MA[ch] = signal.filtfilt(sinc_filter_coeffs, [1.0], interpolated)
        except ValueError:
            df_TL_MA[ch] = np.nan
    trendline_sheet_name = f'Trend line (Sinc Filter C={Sinc_Filter_Cutoff_Period_Hours}h O={filter_order})'

dtdf = df - df_TL_MA
dtdf['Hours'] = df_TL_MA['Hours']
dtdf_5PMA = dtdf.rolling(window=5, center=True, min_periods=1).mean()
dtdf_9PMA = dtdf.rolling(window=9, center=True, min_periods=1).mean()

# ==============================================================================
# EXPORT TO EXCEL & COMPRESSION
# ==============================================================================
print('\nGenerating Excel files...')
note_df = pd.DataFrame.from_dict({
    'A': ['Experiment Number', 'Experiment Title', 'Experiment Start Date', 'TRACES File', '', 'Number of Time Points', 'Total Time Duration (Hours)', 'Average Time Interval (Hours)', 'Detrending Method', 'Trend Line Parameter', '', 'Data Processed Date and Time (UTC)'],
    'E': [Experiment_number, Experiment_title, Date_experiment_started, ttronfilename, '', number_of_rows, total_time, time_interval, Detrending_Method, (f'{Window_size_for_trend_line_moving_average}h window' if Detrending_Method == 'Moving Average' else f'{Sinc_Filter_Cutoff_Period_Hours}h cutoff, {Sinc_Filter_Order} order'), '', datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")]
})

outputexcelfilename = f"{Experiment_number}_data.xlsx"
with pd.ExcelWriter(outputexcelfilename) as writer:
    note_df.to_excel(writer, sheet_name='Note', index=None, header=False)
    df.to_excel(writer, sheet_name='Raw Data')
    df_5PMA.to_excel(writer, sheet_name='5PMA')
    df_9PMA.to_excel(writer, sheet_name='9PMA')
    df_TL_MA.to_excel(writer, sheet_name=trendline_sheet_name)
    dtdf.to_excel(writer, sheet_name='Detrended Data')
    dtdf_5PMA.to_excel(writer, sheet_name='Detrended Data 5PMA')
    dtdf_9PMA.to_excel(writer, sheet_name='Detrended Data 9PMA')

    for k in range(30):
        ch = str(k).zfill(2)
        dfx = pd.DataFrame({
            'Hours': df['Hours'], 'Raw_data': df[ch], '5PMA': df_5PMA[ch], '9PMA': df_9PMA[ch],
            'trend_line': df_TL_MA[ch], 'detrended_data': dtdf[ch], 'detrended_data_5PMA': dtdf_5PMA[ch], 'detrended_data_9MPA': dtdf_9PMA[ch]
        })
        dfx.to_excel(writer, sheet_name=f'Channel {ch}')

    # Peak and Trough generation
    pts = []
    for k in range(30):
        ch = str(k).zfill(2)
        p_idx, t_idx = find_peaks_and_troughs(dtdf_9PMA[ch], distance_in_hours=12, time_interval=time_interval)
        for idx in p_idx: pts.append({'Channel': ch, 'Type': 'Peak', 'Time (Hours)': dtdf_9PMA.loc[idx, 'Hours']})
        for idx in t_idx: pts.append({'Channel': ch, 'Type': 'Trough', 'Time (Hours)': dtdf_9PMA.loc[idx, 'Hours']})
    if pts:
        pd.DataFrame(pts).to_excel(writer, sheet_name='Peaks and Troughs', index=False)

outputexcelfilename2 = f"{Experiment_number}_data_2.xlsx"
with pd.ExcelWriter(outputexcelfilename2) as writer:
    df2.to_excel(writer, sheet_name=excel2_sheet_name, index=None, header=True)

print('\nGenerating .dat files and packing to ZIP...')
df['Days'] = df['Hours'] / 24.0
for k in range(30):
    ch = str(k).zfill(2)
    df.to_csv(f'{ch}.dat', header=False, index=False, sep='\t', columns=['Days', ch])
zip_output_filename = f'{Experiment_number}_data.zip'
!zip -q -r {zip_output_filename} ./*.dat

# ==============================================================================
# PLOT GENERATION & VISUALIZATION
# ==============================================================================
print('\nPlotting summaries...')
plot_output_pdf = f"{Experiment_number}_data_plots.pdf"
x_hours = df['Hours']
x_scale_min = int(math.floor(min(x_hours)/24))*24
x_scale_max = int(math.ceil(max(x_hours)/12))*12+12
xtickslist = list(range(x_scale_min, x_scale_max, mjt))

pp = PdfPages(plot_output_pdf)
plt.rcParams.update({'figure.max_open_warning': 0})

# --- Multi-subplot plots (Raw) ---
fig = plt.figure(figsize=(11, 8.5))
fig.subplots_adjust(hspace=0.15)
fig.suptitle(Experiment_number, fontsize=12)
plt.rc('font', size=5)

for subplot_num, k in enumerate(chlist, 1):
    ch = str(k).zfill(2)
    ax = plt.subplot(subp_raw, subp_col, subplot_num)
    plot_channel_subplot(
        ax, x_hours, df[ch], f'Ch # {ch}', 'blue', df_TL_MA['Hours'], df_TL_MA[ch],
        'trend line', '-r', x_scale_min - 6, x_scale_max, (0, int(max(df[ch]) * 1.1)),
        xtickslist, miton, mit, f'Ch # {ch}'
    )
fig.text(0.50, 0.06, 'Time (hours)', ha='center', fontsize=10)
fig.text(0.08, 0.50, 'Bioluminescence', ha='center', va='center', rotation='vertical', fontsize=10)
pp.savefig(fig)
plt.show()  # Display the raw subplots in Colab output
plt.close(fig)

# --- Multi-subplot plots (Detrended) ---
fig = plt.figure(figsize=(11, 8.5))
fig.subplots_adjust(hspace=0.15)
fig.suptitle(f'{Experiment_number} - detrended data ({Detrending_Method})', fontsize=12)

for subplot_num, k in enumerate(chlist, 1):
    ch = str(k).zfill(2)
    ax = plt.subplot(subp_raw, subp_col, subplot_num)
    plot_channel_subplot(
        ax, dtdf['Hours'], dtdf[ch], f'Ch # {ch}', 'blue', dtdf_5PMA['Hours'], dtdf_5PMA[ch],
        '', '-r', x_scale_min - 6, x_scale_max, None, xtickslist, miton, mit, f'Ch # {ch}'
    )
fig.text(0.50, 0.06, 'Time (hours)', ha='center', fontsize=10)
fig.text(0.08, 0.50, 'Detrended Bioluminescence', ha='center', va='center', rotation='vertical', fontsize=10)
pp.savefig(fig)
plt.show()  # Display the detrended subplots in Colab output
plt.close(fig)

# --- Individual Channel plots + Actogram ---
print('\nGenerating individual channel plots with actograms...')
plt.rc('font', size=10)
for k in chlist:
    plot_individual_channel_summary(
        str(k).zfill(2), df, df_TL_MA, dtdf, dtdf_9PMA, Detrending_Method,
        Sinc_Filter_Cutoff_Period_Hours, filter_order, x_scale_min, x_scale_max,
        xtickslist, miton, mit, sub2label, sub3label, lod, last_time_point, time_interval, pp
    )

# ==============================================================================
# DAMPED SINE FITTING
# ==============================================================================
print('\nPerforming Damped Sine Curve Fitting...')
dtdf_filtered = dtdf[(dtdf['Hours'] >= Start_Hour_for_Fitting) & (dtdf['Hours'] <= End_Hour_for_Fitting)].copy()
all_fit_results = []
plt.rc('font', size=8)

for k in chlist:
    ch = str(k).zfill(2)
    sig_raw = dtdf_filtered[ch].dropna()
    t_raw = dtdf_filtered.loc[sig_raw.index, 'Hours']

    if len(sig_raw) < 5:
        all_fit_results.append({'Channel': ch, 'Fitted Amplitude': np.nan, 'Fitted Period (Hours)': np.nan, 'Fitted Phase (radians)': np.nan, 'Fitted Decay Rate': np.nan, 'Fitted Offset': np.nan})
        continue

    amp_guess = max(0.1, (sig_raw.max() - sig_raw.min()) / 2.0)
    period_guess, decay_guess, offset_guess = 24.0, 0.01, sig_raw.mean()
    t_model = t_raw - t_raw.min()

    try:
        params, _ = curve_fit(
            damped_sine_model, t_model, sig_raw,
            p0=[amp_guess, period_guess, 0.0, decay_guess, offset_guess],
            bounds=([0, 12.0, -2*np.pi, 0, -np.inf], [np.inf, 60.0, 2*np.pi, 0.5, np.inf]),
            maxfev=10000
        )
        norm_phase = params[2] % (2 * np.pi)
        all_fit_results.append({
            'Channel': ch, 'Fitted Amplitude': params[0], 'Fitted Period (Hours)': params[1],
            'Fitted Phase (radians)': norm_phase, 'Fitted Decay Rate': params[3], 'Fitted Offset': params[4]
        })

        # Plot Fitting Curve
        fig, (ax_raw, ax_det) = plt.subplots(2, 1, figsize=(8.5, 11), sharex=True)
        fig.suptitle(f'{Experiment_number} - Damped Sine Fit Channel {ch}', fontsize=12)

        raw_fit = damped_sine_model(t_model, *params) + df_TL_MA.loc[t_raw.index, ch]
        ax_raw.plot(df['Hours'], df[ch], 'o', markersize=2, color='gray', label='Full Raw Data')
        ax_raw.plot(df_TL_MA['Hours'], df_TL_MA[ch], 'k--', linewidth=1.0, label='Trend Line')
        ax_raw.plot(t_raw, raw_fit, 'r-', linewidth=1.5, label='Raw Data Fit')
        ax_raw.set_ylabel('Bioluminescence', fontsize=10)
        ax_raw.legend(loc='upper right', fontsize=8)
        ax_raw.grid(True, linestyle='--', alpha=0.7)

        ax_det.plot(dtdf['Hours'], dtdf[ch], 'o', markersize=2, color='gray', label='Full Detrended Data')
        ax_det.plot(t_raw, sig_raw, 'o', markersize=2, color='blue', label='Fitting Window')
        ax_det.plot(t_raw, damped_sine_model(t_model, *params), 'r-', linewidth=1.5, label='Damped Sine Fit')
        ax_det.set_xlabel('Time (Hours)', fontsize=10)
        ax_det.set_ylabel('Detrended Bioluminescence', fontsize=10)
        ax_det.legend(loc='upper right', fontsize=8)
        ax_det.grid(True, linestyle='--', alpha=0.7)
        ax_det.set_xlim(x_scale_min, x_scale_max)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        pp.savefig(fig)
        plt.show()  # Display the fitting plot in Colab output before closing
        plt.close(fig)

    except Exception as e:
        all_fit_results.append({'Channel': ch, 'Fitted Amplitude': np.nan, 'Fitted Period (Hours)': np.nan, 'Fitted Phase (radians)': np.nan, 'Fitted Decay Rate': np.nan, 'Fitted Offset': np.nan})

pp.close()

df_fit_results = pd.DataFrame(all_fit_results)
new_fitting_excel_filename = f'{Experiment_number}_damped_sine_fit_results.xlsx'
with pd.ExcelWriter(new_fitting_excel_filename) as writer:
    df_fit_results.to_excel(writer, sheet_name='Damped Sine Fit', index=False)

# ==============================================================================
# DOWNLOADING FILES & FINISHING UP
# ==============================================================================
for f_name in [outputexcelfilename, outputexcelfilename2, plot_output_pdf, zip_output_filename, new_fitting_excel_filename]:
    files.download(f_name)

endtime = datetime.now(timezone.utc)
print(f'\nElapsed time: {(endtime - sttime).seconds} seconds')
print(f'Completed at {endtime.strftime("%Y-%m-%d %H:%M:%S")} (UTC)\n')